# Chapter 2: Searching State Spaces

```{admonition} Learning Objectives
:class: tip
- Understand state space representation as graphs
- Implement uninformed search: BFS, DFS, UCS
- Implement informed search: Greedy Best-First, A*
- Apply local search: Hill Climbing, Simulated Annealing
- Solve Constraint Satisfaction Problems: Sudoku, N-Queens, Graph Coloring
- Understand complexity and optimality trade-offs
```

## 2.1 Introduction

Search is fundamental to AI problem-solving. Many problems can be formulated as finding a path through a **state space** from an initial state to a goal state.

### 2.1.1 State Space as a Graph

The state space can be modeled as a directed graph where:
- **Nodes** represent states (configurations)
- **Edges** represent actions (transitions)
- **Path** is a sequence of states connected by actions

**Search Problem Components:**
1. **State Space**: Set of all possible states
2. **Initial State**: Starting configuration
3. **Actions**: Available transitions from each state
4. **Transition Model**: Result of applying an action
5. **Goal Test**: Determines if state satisfies goal
6. **Path Cost**: Total cost of action sequence

In [ ]:
from collections import deque
from typing import List, Set, Optional, Callable, Tuple, Any
import heapq
import random
import math
import time

class SearchProblem:
    def __init__(self, initial_state, goal_state=None):
        self.initial_state = initial_state
        self.goal_state = goal_state
    
    def is_goal(self, state) -> bool:
        return state == self.goal_state
    
    def get_actions(self, state) -> List:
        raise NotImplementedError
    
    def get_successor(self, state, action):
        raise NotImplementedError
    
    def action_cost(self, state, action, next_state) -> float:
        return 1.0

print('Search problem framework defined')

## 2.2 Breadth-First Search

BFS explores all nodes at depth d before nodes at depth d+1.

**Properties:**
- Complete: Yes
- Optimal: Yes (for uniform cost)
- Time: O(b^d)
- Space: O(b^d)

In [ ]:
def breadth_first_search(problem, verbose=True):
    start_time = time.time()
    frontier = deque([(problem.initial_state, [])])
    visited = {problem.initial_state}
    nodes_expanded = 0
    
    while frontier:
        state, path = frontier.popleft()
        nodes_expanded += 1
        
        if problem.is_goal(state):
            if verbose:
                print(f'BFS: Goal found in {time.time()-start_time:.4f}s')
                print(f'Nodes expanded: {nodes_expanded}')
            return path + [state]
        
        for action in problem.get_actions(state):
            next_state = problem.get_successor(state, action)
            if next_state not in visited:
                visited.add(next_state)
                frontier.append((next_state, path + [state]))
    
    return None

print('BFS implemented')

## 2.3 A* Search

A* combines path cost g(n) and heuristic h(n):

$$f(n) = g(n) + h(n)$$

Optimal if h(n) is admissible (never overestimates).

In [ ]:
def a_star_search(problem, heuristic, verbose=True):
    start_time = time.time()
    counter = 0
    h0 = heuristic(problem.initial_state)
    frontier = [(h0, counter, 0, problem.initial_state, [])]
    visited = {}
    nodes_expanded = 0
    
    while frontier:
        f, _, g, state, path = heapq.heappop(frontier)
        
        if state in visited and visited[state] <= g:
            continue
        
        visited[state] = g
        nodes_expanded += 1
        
        if problem.is_goal(state):
            if verbose:
                print(f'A*: Goal found with cost {g:.2f}')
                print(f'Nodes expanded: {nodes_expanded}')
                print(f'Time: {time.time()-start_time:.4f}s')
            return path + [state]
        
        for action in problem.get_actions(state):
            next_state = problem.get_successor(state, action)
            new_g = g + problem.action_cost(state, action, next_state)
            
            if next_state not in visited or new_g < visited.get(next_state, float('inf')):
                h = heuristic(next_state)
                new_f = new_g + h
                counter += 1
                heapq.heappush(frontier, (new_f, counter, new_g, next_state, path + [state]))
    
    return None

print('A* Search implemented')

## 2.4 8-Puzzle Example

Classic sliding puzzle problem.

In [ ]:
class EightPuzzle(SearchProblem):
    def __init__(self, initial, goal=None):
        if goal is None:
            goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)
        super().__init__(tuple(initial), tuple(goal))
    
    def find_blank(self, state):
        return state.index(0)
    
    def get_actions(self, state):
        blank = self.find_blank(state)
        row, col = blank // 3, blank % 3
        actions = []
        if row > 0: actions.append('UP')
        if row < 2: actions.append('DOWN')
        if col > 0: actions.append('LEFT')
        if col < 2: actions.append('RIGHT')
        return actions
    
    def get_successor(self, state, action):
        blank = self.find_blank(state)
        row, col = blank // 3, blank % 3
        moves = {'UP': (-1, 0), 'DOWN': (1, 0), 'LEFT': (0, -1), 'RIGHT': (0, 1)}
        dr, dc = moves[action]
        new_row, new_col = row + dr, col + dc
        new_blank = new_row * 3 + new_col
        state_list = list(state)
        state_list[blank], state_list[new_blank] = state_list[new_blank], state_list[blank]
        return tuple(state_list)

def manhattan_distance(state):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            goal_row = (state[i] - 1) // 3
            goal_col = (state[i] - 1) % 3
            curr_row = i // 3
            curr_col = i % 3
            distance += abs(goal_row - curr_row) + abs(goal_col - curr_col)
    return distance

print('8-Puzzle implementation complete')

In [ ]:
initial = (1, 2, 3, 4, 0, 6, 7, 5, 8)
puzzle = EightPuzzle(initial)
print('Testing A* on 8-puzzle...')
solution = a_star_search(puzzle, manhattan_distance)
if solution:
    print(f'Solution found with {len(solution)} steps')

## 2.5 Sudoku Solver

Constraint Satisfaction Problem using backtracking.

In [ ]:
def solve_sudoku(grid):
    def is_valid(grid, row, col, num):
        if num in grid[row]:
            return False
        if num in [grid[i][col] for i in range(9)]:
            return False
        box_row, box_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(box_row, box_row + 3):
            for j in range(box_col, box_col + 3):
                if grid[i][j] == num:
                    return False
        return True
    
    def solve():
        for row in range(9):
            for col in range(9):
                if grid[row][col] == 0:
                    for num in range(1, 10):
                        if is_valid(grid, row, col, num):
                            grid[row][col] = num
                            if solve():
                                return True
                            grid[row][col] = 0
                    return False
        return True
    
    solve()
    return grid

puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

solution = solve_sudoku([row[:] for row in puzzle])
print('Sudoku solved!')

## 2.6 N-Queens Problem

Place N queens on NxN board so none attack each other.

In [ ]:
def solve_n_queens(n=8):
    def is_safe(board, row, col):
        for i in range(row):
            if board[i] == col:
                return False
            if abs(board[i] - col) == abs(i - row):
                return False
        return True
    
    def backtrack(board, row):
        if row == n:
            solutions.append(board[:])
            return
        for col in range(n):
            if is_safe(board, row, col):
                board[row] = col
                backtrack(board, row + 1)
                board[row] = -1
    
    solutions = []
    backtrack([-1] * n, 0)
    return solutions

solutions = solve_n_queens(8)
print(f'Found {len(solutions)} solutions for 8-Queens')

## 2.7 Summary

### Key Algorithms

| Algorithm | Complete | Optimal | Time | Space |
|-----------|----------|---------|------|-------|
| BFS | Yes | Yes | O(b^d) | O(b^d) |
| DFS | No | No | O(b^m) | O(bm) |
| A* | Yes | Yes | O(b^d) | O(b^d) |

A* is optimal when using admissible heuristics.

### CSP Techniques
- Backtracking search
- Constraint propagation
- Variable and value ordering heuristics

---

**Next**: [Chapter 3 - Multiagent Search](ch03_multiagent.ipynb)